# Qwen3-4B QLoRA classification → Sailor2-8B QLoRA correction

Enable Internet and **GPU T4 x2**. Both stages use NF4 double-quantized QLoRA, sequentially on one GPU by default. Unsupported BF16 GPUs such as T4 automatically use FP16; preflight prints the actual dtype and checks a conservative correction memory budget. Correction also supports explicit two-GPU balanced placement, with no DDP or CPU/disk offload. QLoRA reduces base-weight memory but does not guarantee Sailor2 training fits an 8GB GPU.

Attach a **private** dataset containing `VGEC 2026_v1.json`, `VGEC_classifier.json` and `VGEC_balanced.json`. The classifier trains on all original training labels plus capped synthetic examples, with class weights for residual imbalance; the corrector uses its separate audited bundle. Both share the original holdout. Training correction hints come from train-only group-safe mixed-data out-of-fold classifiers; validation hints come from the saved main classifier. Predicted class 0 never skips correction.

Publish the code branch before running. Two separate JSON configs control the tasks, both with seed 42. Kaggle overrides clear repository-local dataset paths so the attached private files are discovered under `/kaggle/input`. No Kaggle credentials or dataset are published by this notebook.

## Alpha: head/LoRA learning-rate ablation

Baseline `rquq/test` uses LR 2e-5 for both LoRA and its newly initialized
classification head. This branch uses `training.head_learning_rate: 1e-4`
for the head and `training.args.learning_rate: 2e-5` for LoRA. Both groups
use the same linear schedule and existing decay policy; rates are recorded
in classifier/run metadata. All other classifier settings remain unchanged.
For a matched classifier-only comparison, use `--skip-hints` with each branch's
classification CLI; do not reuse those runs as correction-hint artifacts.
Minimal pairs and ensembling are intentionally deferred.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import signal
import sys

# The baseline uses PyTorch; avoid importing optional TensorFlow/Flax stacks.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")

REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "rquq/alpha"  # Separate-head-LR experiment; baseline is rquq/test.
PROJECT_SUBDIR = "."  # The baseline lives at the root of this repository.
CLASSIFICATION_CONFIG = "configs/classification.json"
CORRECTION_CONFIG = "configs/correction.json"
CLASSIFICATION_OVERRIDES = {"run": {"seed": 42}, "data": {"path": None, "augmentation_path": None}}
CORRECTION_OVERRIDES = {"run": {"seed": 42}, "data": {"path": None}}
CLASSIFIER_RUN = None  # Optional Qwen3 run with classifier/{classifier_config,adapter_config}.json and hints.json.
USE_HF_SECRET = False
CLONE_DIR = Path("/kaggle/working/gecc-alpha-source")
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
print("Branch:", BRANCH, "| configs:", CLASSIFICATION_CONFIG, CORRECTION_CONFIG)



In [ ]:
# This cell can be rerun after a kernel restart. Git runs as a subprocess.
# Reuse only a clean checkout of the same repository and branch.
from urllib.parse import urlsplit
if urlsplit(REPO_URL).username is not None:
    raise ValueError("Do not embed GitHub credentials in REPO_URL.")
subprocess.run(["git", "check-ref-format", "--branch", BRANCH], check=True, capture_output=True)
if CLONE_DIR.exists():
    if not (CLONE_DIR / ".git").is_dir():
        raise RuntimeError(f"{CLONE_DIR} exists but is not a Git clone; choose another CLONE_DIR.")
    existing_url = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=CLONE_DIR, text=True).strip()
    if existing_url != REPO_URL:
        raise RuntimeError("CLONE_DIR belongs to a different repository; choose another CLONE_DIR.")
    existing_branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=CLONE_DIR, text=True).strip()
    if existing_branch != BRANCH:
        raise RuntimeError("CLONE_DIR uses a different branch; choose another CLONE_DIR.")
    changes = subprocess.check_output(["git", "status", "--porcelain"], cwd=CLONE_DIR, text=True)
    if changes.strip():
        raise RuntimeError("CLONE_DIR has local changes; preserve them and choose another CLONE_DIR.")

remote = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", "--", REPO_URL,
                         f"refs/heads/{BRANCH}"], capture_output=True, text=True)
if remote.returncode == 2:
    raise RuntimeError(f"Branch {BRANCH!r} does not exist on GitHub. Push it or select a published BRANCH.")
if remote.returncode != 0:
    raise RuntimeError("Cannot read the Git repository. Enable Kaggle Internet and check repository access.")
if CLONE_DIR.exists():
    subprocess.run(["git", "fetch", "origin", f"refs/heads/{BRANCH}"], cwd=CLONE_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", "FETCH_HEAD"], cwd=CLONE_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
                    "--", REPO_URL, str(CLONE_DIR)], check=True)
PROJECT_DIR = (CLONE_DIR / PROJECT_SUBDIR).resolve()
if not PROJECT_DIR.is_relative_to(CLONE_DIR.resolve()):
    raise ValueError("PROJECT_SUBDIR must stay inside the cloned repository.")
for filename in ["requirements.txt", "model/classification.py", "model/correction.py", "model/two_phase.py", CLASSIFICATION_CONFIG, CORRECTION_CONFIG]:
    if not (PROJECT_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {PROJECT_DIR}; check the selected branch and PROJECT_SUBDIR.")
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH, "project_subdir": PROJECT_SUBDIR,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CLONE_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))



In [ ]:
# Keep the CUDA-enabled torch provided by Kaggle. A constraint prevents
# pip from replacing it while installing the pinned Hugging Face stack.
from importlib.metadata import version
from packaging.version import Version
import tempfile

installed_torch = version("torch")
if Version(installed_torch).base_version != "2.7.1":
    raise RuntimeError(f"requirements.txt targets the HiGANplus stack with CUDA PyTorch 2.7.1; found {installed_torch}. Use a matching image rather than replacing its CUDA build.")
packages = ["transformers", "peft", "accelerate", "safetensors", "huggingface-hub", "tokenizers", "bitsandbytes"]
imported_before = {name: version(name) for name in packages if name.replace("-", "_") in sys.modules}
with tempfile.TemporaryDirectory() as temporary:
    constraints = Path(temporary) / "constraints.txt"
    constraints.write_text(f"torch=={installed_torch}\n", encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt"),
                    "--constraint", str(constraints)], cwd=PROJECT_DIR, check=True)
changed_imports = [name for name, old in imported_before.items() if version(name) != old]
if changed_imports:
    raise RuntimeError(f"Restart the kernel to reload {changed_imports}, then rerun from the parameter cell; the clean clone is reused.")
# Check the training stack directly; unrelated packages in Kaggle's image
# can have pre-existing conflicts that do not affect this PyTorch baseline.
subprocess.run([sys.executable, "-c", "import torch, transformers, peft, accelerate, bitsandbytes; print('Training imports passed')"], check=True)
print("Pinned dependencies installed; retained torch", installed_torch)



In [ ]:
if USE_HF_SECRET:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

def apply_overrides(target, overrides):
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            apply_overrides(target[key], value)
        else:
            target[key] = value

config_paths = []
for name, filename, overrides in [("classification", CLASSIFICATION_CONFIG, CLASSIFICATION_OVERRIDES),
                                  ("correction", CORRECTION_CONFIG, CORRECTION_OVERRIDES)]:
    config = json.loads((PROJECT_DIR / filename).read_text(encoding="utf-8"))
    apply_overrides(config, overrides)
    path = CLONE_DIR.parent / f"{name}-{RUN_ID}.json"
    path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    config_paths.append(path)
OUTPUT_DIR = PROJECT_DIR / "runs" / f"qwen3-sailor2-{RUN_ID}"
command = [sys.executable, "-m", "model.two_phase",
           "--classification-config", str(config_paths[0]),
           "--correction-config", str(config_paths[1]),
           "--output-dir", str(OUTPUT_DIR)]
if CLASSIFIER_RUN:
    command += ["--classifier-run", str(CLASSIFIER_RUN)]
# Check split compatibility, classifier CUDA/dtype, and correction GPU free memory before weight downloads.
subprocess.run(command + ["--preflight-only"], cwd=PROJECT_DIR, check=True)


In [ ]:
# Train the Qwen3 QLoRA classifier and score head, build OOF hints, then train Sailor2 QLoRA.
# Separate subprocesses release classifier memory before correction training.
LOG_PATH = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}.log"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
with LOG_PATH.open("w", encoding="utf-8") as log_file:
    log_file.write(json.dumps(SOURCE_INFO, ensure_ascii=False) + "\n")
    log_file.flush()
    with subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log_file.write(line)
                log_file.flush()
            returncode = process.wait()
        except BaseException:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


In [ ]:
from IPython.display import JSON, FileLink, display
for filename in ["pipeline_manifest.json", "source_manifest.json"]:
    print(filename)
    display(JSON(json.loads((OUTPUT_DIR / filename).read_text(encoding="utf-8"))))
archive_path = Path(str(OUTPUT_DIR) + "-artifacts.zip")
if archive_path.is_file():
    display(FileLink(str(archive_path)))
if LOG_PATH.is_file():
    display(FileLink(str(LOG_PATH)))
print("Qwen3 QLoRA classifier:", OUTPUT_DIR / "classification" / "classifier")
print("Sailor2 NF4 QLoRA adapter:", OUTPUT_DIR / "correction" / "adapter")
print("Save a notebook version to retain private run outputs.")


## Saved model use

Each task retains its resolved config, training/validation IDs, predictions, metrics, and manifest. The classifier saves its tokenizer, `preprocessing.json`, `classifier_config.json` (exact base revision and actual quantization/dtype), and the LoRA adapter with the five-class score head. `classification/hints.json` includes OOF fold membership. `pipeline_manifest.json` connects the classifier and correction adapter. Historical PhoBERT runs are not reusable Qwen3 classifiers. Checkpoints are selected on the holdout, not an independent test set; no quality gain is guaranteed.

```bash
python -m model.pipeline_predict --classifier RUN/classification/classifier \
  --adapter RUN/correction/adapter --data INPUT.json --output predictions.json
```

Supply `--evaluate` only when gold outputs exist. Output formatting remains a local proposal pending organizer instructions. For local classification only, use `python -m model.classification --config configs/classification.json`. Both default NF4 QLoRA stages require CUDA. Correction saves its resolved dtype, quantization settings and prompt beside the adapter and reloads them for inference.
